# VAECox — Full Reproduction on Real TCGA Data (Kaggle GPU)

**Reproducibility study of** *Kim, Kim, Choe, Lee & Kang (2020),
"Improved survival analysis by learning shared genomic information from
pan-cancer data", Bioinformatics 36(Suppl_1):i389–i398.*
DOI: 10.1093/bioinformatics/btaa462 · Code: https://github.com/dmis-lab/VAECox

This single notebook runs the **entire** pipeline end-to-end:

1. Read real TCGA RNA-seq + survival from the attached **GenoTEX** dataset
   (`input/TCGA/` — sourced from UCSC Xena, open access, no dbGaP).
2. Preprocess (expression is already log2 → per-gene z-normalisation).
3. Pretrain the **VAE** on pan-cancer expression (GPU).
4. Train + evaluate all survival models, including the **fine-tuned VAECox**
   (the paper's actual method — encoder is *unfrozen*).
5. Reproduce the headline claim: **C-index, VAECox vs baselines on 10 cancers**.
6. Run the extensions: robustness, fairness, lightweight models, feature importance, Kaplan–Meier.
7. Write all result CSVs, figures, and a reproducibility card to `/kaggle/working`.

### How to run on Kaggle
* Add data → attach **GenoTEX: LLM Agent Benchmark for Genomic Analysis** (haoyangliu14).
* Settings → Accelerator → **GPU T4 x2** (or P100). Internet can be **Off** (data is local).
* Run all cells. Checkpoints + results land in `/kaggle/working` and persist as notebook output.
* The heavy cells (VAE pretrain, Phase 2) print progress and save intermediate files, so a
  12-hour session timeout never loses completed work — just re-run and it resumes from cache.

> **DATA NOTE (read once):** the loader auto-discovers `input/TCGA/` under `/kaggle/input`,
> extracts the cohort code from each filename (`TCGA.BLCA.sampleMap_...`), and pulls overall
> survival from the `clinicalMatrix`. Check the printed per-cohort event counts — they should be
> in the dozens–hundreds (real cohorts), not single digits (toy data).

## 0 · Config & environment

In [ ]:
# Kaggle's base image lacks lifelines — install it before anything imports it.
try:
    import lifelines  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lifelines"], check=True)
    print("installed lifelines")

In [ ]:
import os, sys, gc, io, gzip, time, json, math, urllib.request, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device = {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ---- Reproducibility config (mirrors the paper / repo) ----------------------
CFG = dict(
    # 10 cancers evaluated in the paper (Table 1). Whichever of these are present
    # in the attached data get evaluated; ALL loaded cohorts feed VAE pretraining.
    PAPER_10   = ["BLCA", "BRCA", "HNSC", "KIRC", "LGG",
                  "LIHC", "LUAD", "LUSC", "OV", "STAD"],
    HIDDEN     = 4096,     # VAE hidden layer  (paper)
    LATENT     = 128,      # VAE latent dim    (paper)
    VAE_EPOCHS = 500,      # paper: 500  (set to 50 for a fast smoke-test)
    VAE_LR     = 1e-3,
    VAE_WD     = 1e-5,
    VAE_BATCH  = 256,      # minibatch for GPU efficiency (paper used full batch on GPU)
    SURV_EPOCHS= 100,
    SEEDS      = list(range(10)),   # paper: 10 seeds
    HP_SEARCH  = True,     # reduced grid, searched once per cancer (see §5)
    OUT        = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./out",
    # §6a/§6c are single-cohort versions of experiments that §6i/§6j redo across
    # every cohort. Running both wastes ~40-60 min for no extra result.
    SKIP_SUPERSEDED = True,
    RESUME     = True,     # reuse CSVs from a previous run instead of recomputing
    # Report 1/2 both flagged that the pan-cancer VAE pretraining pool includes
    # the test-fold patients of the 10 evaluation cohorts, so the encoder being
    # credited for VAECox's advantage has seen those patients' expression before
    # any split is drawn. True (default) excludes, per eval cohort, the UNION of
    # every seed's test fold from the pretraining matrix; False reproduces the
    # original (leaky) behaviour for an A/B comparison. See §1 pancancer_matrix().
    PRETRAIN_EXCLUDE_TEST = True,
)
os.makedirs(CFG["OUT"], exist_ok=True)
os.makedirs(f'{CFG["OUT"]}/results', exist_ok=True)
os.makedirs(f'{CFG["OUT"]}/figures', exist_ok=True)


def set_seed(s):
    np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


print("Cancers to evaluate (if present in data):", CFG["PAPER_10"])


def find_prev(filename):
    """Locate a result file from THIS session or from a previous run attached as
    an input dataset. Kaggle sessions end (12h cap, GPU quota), so every
    expensive stage is resumable rather than all-or-nothing."""
    local = f'{CFG["OUT"]}/results/{filename}'
    if os.path.exists(local):
        return local
    import glob as _g
    hits = _g.glob(f"/kaggle/input/**/{filename}", recursive=True)
    return hits[0] if hits else None


def cached(name, builder):
    """Return `builder()`'s DataFrame, or a previous run's CSV if one exists.

    Each builder already writes results/<name>.csv as its last act, so reloading
    that file is equivalent to re-running it — minus the hours.
    """
    fn = f"{name}.csv"
    prev = find_prev(fn) if CFG["RESUME"] else None
    if prev:
        df = pd.read_csv(prev)
        dest = f'{CFG["OUT"]}/results/{fn}'
        if os.path.abspath(prev) != os.path.abspath(dest):
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            df.to_csv(dest, index=False)
        print(f"  [resume] {fn}: {len(df)} rows from {prev} — not recomputed")
        return df
    try:
        return builder()
    except Exception:
        # A Kaggle run that raises publishes NO output at all, so one broken
        # stage would discard every result computed before it. Log and continue;
        # downstream cells already handle an empty frame, and §9 reports the gap.
        import traceback
        print(f"  !! {name} FAILED — continuing so earlier results still save")
        traceback.print_exc()
        return pd.DataFrame()


## 1 · Load real TCGA data from the attached GenoTEX dataset

The GenoTEX benchmark ships `input/TCGA/`, downloaded directly from the UCSC
Xena TCGA Hub: one folder per cancer, each with a `HiSeqV2_PANCAN` expression
matrix (already log2, pan-cancer normalised) and a `clinicalMatrix` holding
survival. We auto-discover the folder, read the cohort code from the filename
(`TCGA.BLCA.sampleMap_...`), and extract overall survival (time + event).

No internet needed — everything is mounted read-only under `/kaggle/input`.

In [ ]:
import glob, re

# Locate the GenoTEX TCGA folder. The mount path depends on how the dataset was
# attached, so try the known locations first and only then fall back to a wide
# recursive search (slow — GenoTEX also ships code/, imgs/, output/).
_KNOWN = [
    "/kaggle/input/datasets/haoyangliu14/genotex-llm-agent-benchmark-for-genomic-analysis/input/TCGA",
    "/kaggle/input/genotex-llm-agent-benchmark-for-genomic-analysis/input/TCGA",
    "/kaggle/input/genotex/input/TCGA",
]
_cands = [q for q in _KNOWN if os.path.isdir(q)]
if not _cands:
    _cands = (glob.glob("/kaggle/input/*/*/*/input/TCGA") or
              glob.glob("/kaggle/input/*/input/TCGA") or
              glob.glob("/kaggle/input/**/input/TCGA", recursive=True) or
              glob.glob("/kaggle/input/**/TCGA", recursive=True) or
              glob.glob("./**/input/TCGA", recursive=True))
if not _cands:
    # Show what actually IS mounted — far more useful than a bare assertion.
    print("Could not find the GenoTEX TCGA folder. Mounted under /kaggle/input:")
    for _r, _d, _f in os.walk("/kaggle/input"):
        _depth = _r.rstrip("/").count("/") - 2
        if _depth > 3:
            _d[:] = []
            continue
        print("   " * _depth + os.path.basename(_r) + "/")
    raise FileNotFoundError(
        "Add Data -> search 'GenoTEX' -> attach 'GenoTEX: LLM Agent Benchmark for "
        "Genomic Analysis'. If it is attached under a different name, set TCGA_ROOT "
        "manually to the folder holding the per-cancer subfolders.")
TCGA_ROOT = _cands[0]
_n_folders = len([d for d in os.listdir(TCGA_ROOT)
                  if os.path.isdir(os.path.join(TCGA_ROOT, d))])
print(f"TCGA_ROOT = {TCGA_ROOT}  ({_n_folders} cohort folders)")


def _resolve(path):
    """Kaggle unzips .gz files into a *directory* — descend to the real data file."""
    if os.path.isfile(path):
        return path
    if os.path.isdir(path):
        best, best_sz = None, -1
        for r, _, fs in os.walk(path):
            for fn in fs:
                p = os.path.join(r, fn)
                sz = os.path.getsize(p)
                if sz > best_sz:
                    best, best_sz = p, sz
        return best
    return None


def _read_tsv(path):
    """Read a Xena TSV; detect gzip by magic bytes (filename may lack .gz)."""
    path = _resolve(path)
    with open(path, "rb") as fh:
        gzipped = fh.read(2) == b"\x1f\x8b"
    if gzipped:
        with gzip.open(path, "rt") as f:
            return pd.read_csv(f, sep="\t", index_col=0)
    return pd.read_csv(path, sep="\t", index_col=0)


def parse_survival(clin):
    """Extract overall survival from a Xena clinicalMatrix.
       Returns DataFrame(index=sample) with 'survival' (days) + 'censored' (0=event,1=censored)."""
    C = {c.lower(): c for c in clin.columns}
    # 1) explicit OS time + event indicator (several Xena vintages)
    for tcol, ecol in [("os.time", "os"), ("_os", "_os_ind"),
                       ("_time_to_event", "_event"), ("os_time", "os_status")]:
        if tcol in C and ecol in C:
            t = pd.to_numeric(clin[C[tcol]], errors="coerce")
            e = pd.to_numeric(clin[C[ecol]], errors="coerce")
            df = pd.DataFrame({"survival": t, "censored": (1 - e)}).dropna()
            if len(df) > 10:
                df["censored"] = df["censored"].astype(int)
                return df
    # 2) derive from vital_status + days_to_death / days_to_last_followup
    if "vital_status" in C:
        vs = clin[C["vital_status"]].astype(str).str.upper()
        dead = vs.str.startswith("DEAD") | vs.str.contains("DECEAS")
        dtd = pd.to_numeric(clin[C["days_to_death"]], errors="coerce") \
              if "days_to_death" in C else pd.Series(np.nan, index=clin.index)
        dtf = pd.to_numeric(clin[C["days_to_last_followup"]], errors="coerce") \
              if "days_to_last_followup" in C else pd.Series(np.nan, index=clin.index)
        surv = np.where(dead, dtd, dtf)
        df = pd.DataFrame({"survival": surv, "censored": (~dead).astype(int)},
                          index=clin.index).dropna()
        return df
    return None


def cohort_code(folder, exp_f=None):
    """Recover the TCGA cohort code (BLCA, STAD, ...).

    Xena filenames carry it ('TCGA.BLCA.sampleMap_HiSeqV2_PANCAN'); GenoTEX names
    its folders 'TCGA_Bladder_Cancer_(BLCA)', so fall back to the parenthesised
    suffix — uppercasing the whole folder name would yield a code that matches no
    entry in PAPER_10 and would silently drop the cohort from evaluation.
    """
    m = re.search(r"TCGA\.([A-Za-z]+)\.sampleMap", exp_f or "")
    if m:
        return m.group(1).upper()
    m = re.search(r"\(([A-Za-z]+)\)\s*$", os.path.basename(folder).strip())
    if m:
        return m.group(1).upper()
    return os.path.basename(folder).upper()


def load_genotex_cohort(folder):
    """Return (cohort_code, DataFrame[genes + survival + censored]) or (code, None)."""
    files = os.listdir(folder)
    exp_f = next((f for f in files if "HiSeqV2_PANCAN" in f), None) or \
            next((f for f in files if "HiSeqV2" in f), None)
    cli_f = next((f for f in files if "clinicalMatrix" in f), None)
    if not exp_f or not cli_f:
        return cohort_code(folder), None
    code = cohort_code(folder, exp_f)

    expr = _read_tsv(os.path.join(folder, exp_f)).T          # samples x genes
    expr.index = expr.index.astype(str).str[:15]
    expr = expr[~expr.index.duplicated(keep="first")]

    surv = parse_survival(_read_tsv(os.path.join(folder, cli_f)))
    if surv is None:
        return code, None
    surv.index = surv.index.astype(str).str[:15]
    surv = surv[~surv.index.duplicated(keep="first")]

    common = [s for s in expr.index.intersection(surv.index) if s[13:15] == "01"]
    if len(common) < 20:
        return code, None
    df = expr.loc[common].dropna(axis=1)
    df["survival"] = surv.loc[common, "survival"].astype(float).values
    df["censored"] = surv.loc[common, "censored"].astype(int).values
    return code, df[df["survival"] > 0]


# ---------- Load every cohort folder ----------
COHORT_DFS = {}
for folder in sorted(glob.glob(os.path.join(TCGA_ROOT, "*"))):
    if not os.path.isdir(folder):
        continue
    code, df = load_genotex_cohort(folder)
    if df is not None:
        COHORT_DFS[code] = df
    else:
        print(f"  skip {code} (no usable expression/survival)")

if not COHORT_DFS:
    raise RuntimeError("No usable TCGA cohorts loaded from GenoTEX.")

# Genes shared across all loaded cohorts → consistent VAE input dim.
GENES = sorted(set.intersection(*[set(d.columns) - {"survival", "censored"}
                                  for d in COHORT_DFS.values()]))
NUM_FEATURES = len(GENES)

# Evaluate whichever of the paper's 10 are present; all cohorts feed VAE pretraining.
CFG["PAPER_10"] = [c for c in CFG["PAPER_10"] if c in COHORT_DFS]
DATA_SOURCE = "genotex"

print(f"\nDATA SOURCE = REAL TCGA (GenoTEX / UCSC Xena)")
print(f"Cohorts loaded ({len(COHORT_DFS)}): {sorted(COHORT_DFS)}")
print(f"Evaluating (paper 10 present): {CFG['PAPER_10']}")
print(f"Shared genes (VAE input dim): {NUM_FEATURES}")
for c in sorted(COHORT_DFS):
    d = COHORT_DFS[c]
    n_ev = int((d['censored'] == 0).sum())
    print(f"  {c:6s}: N={len(d):4d}  events={n_ev:4d}  censor%={100*(d['censored']==1).mean():.0f}")

## 2 · Preprocessing & splits

Per-gene **z-normalisation fit on the training set only** (no leakage), and a
stratified 80/20 split by survival-time quintile — matching the repo's Phase 1.

The pan-cancer matrix used to pretrain the VAE (`pancancer_matrix()` below) also
excludes, for the 10 evaluation cohorts only, every seed's test-fold patients
(`CFG["PRETRAIN_EXCLUDE_TEST"]`, on by default) — otherwise the encoder sees the
expression of patients it is later scored on. Set the flag to `False` to
reproduce the original (leaky) pretraining pool for comparison.


In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler


def cohort_matrix(cohort):
    d = COHORT_DFS[cohort]
    X = d[GENES].values.astype(np.float32)
    y = d["survival"].values.astype(np.float64)
    c = d["censored"].values.astype(np.int32)
    return X, y, c


def make_split(cohort, seed):
    X, y, c = cohort_matrix(cohort)
    # stratify by survival quintile (fallback to event indicator if too few)
    try:
        strata = pd.qcut(y, q=min(5, len(np.unique(y))), labels=False, duplicates="drop")
    except Exception:
        strata = c
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    tr, te = next(sss.split(X, strata))
    sc = StandardScaler().fit(X[tr])
    return (sc.transform(X[tr]).astype(np.float32), sc.transform(X[te]).astype(np.float32),
            y[tr], y[te], c[tr], c[te])


def _held_out_test_rows(cohort, seeds):
    """Row indices of `cohort` that land in the test fold of ANY of `seeds`.

    Used to keep every evaluation-seed's test patients out of VAE pretraining —
    otherwise the pretrained encoder has seen those patients' raw expression
    before the split that later scores it on them (flagged in both reviews).
    """
    X, y, c = cohort_matrix(cohort)
    try:
        strata = pd.qcut(y, q=min(5, len(np.unique(y))), labels=False, duplicates="drop")
    except Exception:
        strata = c
    held = set()
    for seed in seeds:
        sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
        _, te = next(sss.split(X, strata))
        held.update(te.tolist())
    return held


def pancancer_matrix(exclude_test=None, seeds=None):
    """Stacked z-normalised expression across ALL cohorts for VAE pretraining.

    `exclude_test` (default: CFG["PRETRAIN_EXCLUDE_TEST"]) drops, for each of the
    10 evaluation cohorts only, the union of every seed's test-fold patients
    before pretraining — cohorts that are pretraining-only (never evaluated) are
    unaffected. This is a single pretraining run rather than one per seed (which
    would close the gap completely but multiply VAE training time by len(seeds));
    it still guarantees no evaluation-seed's test patient was seen during
    pretraining. Set exclude_test=False to reproduce the original behaviour.
    """
    exclude_test = CFG["PRETRAIN_EXCLUDE_TEST"] if exclude_test is None else exclude_test
    seeds = seeds if seeds is not None else CFG["SEEDS"]
    mats, n_dropped = [], {}
    for c in COHORT_DFS:
        X, y, cens = cohort_matrix(c)
        if exclude_test and c in CFG["PAPER_10"]:
            held = _held_out_test_rows(c, seeds)
            keep = np.array([i for i in range(len(X)) if i not in held])
            if len(keep) >= 10:
                n_dropped[c] = len(X) - len(keep)
                X = X[keep]
            else:
                n_dropped[c] = 0  # cohort too small to drop anything safely
        mats.append(StandardScaler().fit_transform(X).astype(np.float32))
    if exclude_test and n_dropped:
        total = sum(n_dropped.values())
        print(f"pancancer_matrix: excluded {total} eval-cohort test patients "
              f"(union over {len(seeds)} seeds) from VAE pretraining")
    return np.vstack(mats)


X_PAN = pancancer_matrix()
print(f"Pan-cancer VAE matrix: {X_PAN.shape}  ({X_PAN.nbytes/1e6:.0f} MB)"
      f"  [test-leakage {'excluded' if CFG['PRETRAIN_EXCLUDE_TEST'] else 'NOT excluded (legacy)'}]")


## 3 · VAE model (faithful to the repo's `vae_models.VAE`)

Encoder `p → 4096 → (μ,σ) 128`, decoder `128 → 4096 → p`, Tanh activations,
loss = MSE reconstruction + KL divergence.

In [ ]:
class VAE(nn.Module):
    def __init__(self, num_features, hidden=4096, latent=128, dropout=0.0):
        super().__init__()
        self.encode = nn.Sequential(nn.Linear(num_features, hidden), nn.Tanh(), nn.Dropout(dropout))
        self.encode_mu = nn.Sequential(nn.Linear(hidden, latent), nn.Tanh(), nn.Dropout(dropout))
        self.encode_si = nn.Sequential(nn.Linear(hidden, latent), nn.Tanh(), nn.Dropout(dropout))
        self.decode = nn.Sequential(nn.Linear(latent, hidden), nn.Tanh(), nn.Dropout(dropout),
                                    nn.Linear(hidden, num_features))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)

    def reparam(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def embed(self, x):
        return self.encode_mu(self.encode(x))

    def forward(self, x):
        h = self.encode(x)
        mu, logvar = self.encode_mu(h), self.encode_si(h)
        recon = self.decode(mu)
        mse = F.mse_loss(recon, x, reduction="mean")
        kld = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
        return mse + kld


# Named by pretraining-data regime so flipping PRETRAIN_EXCLUDE_TEST can never
# silently reuse a checkpoint trained under the other (leaky vs. leak-free) pool.
VAE_CKPT_NAME = "vae_pretrained_leakfree.pt" if CFG["PRETRAIN_EXCLUDE_TEST"] else "vae_pretrained.pt"


def _find_pretrained():
    """Look for a pretrained VAE: working-dir cache first, then any attached dataset."""
    local = f'{CFG["OUT"]}/{VAE_CKPT_NAME}'
    if os.path.exists(local):
        return local
    hits = (glob.glob(f"/kaggle/input/**/{VAE_CKPT_NAME}", recursive=True) or
            glob.glob("/kaggle/input/**/*.pt", recursive=True))
    return hits[0] if hits else None


def train_vae():
    ckpt = _find_pretrained()
    if ckpt:
        print(f"Loading pretrained VAE (skipping training): {ckpt}")
        vae = VAE(NUM_FEATURES, CFG["HIDDEN"], CFG["LATENT"]).to(DEVICE)
        try:
            vae.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        except RuntimeError as e:
            raise RuntimeError(
                f"Checkpoint shape mismatch — it was trained with a different "
                f"gene set than the current NUM_FEATURES={NUM_FEATURES}. "
                f"Delete the attached .pt to retrain, or re-use the exact data. "
                f"Original error: {e}")
        return vae
    # No checkpoint found, so train one. `ckpt` is None here — the save path has
    # to be set explicitly, and it must be somewhere writable (/kaggle/input is
    # read-only, /kaggle/working is not).
    ckpt = f'{CFG["OUT"]}/{VAE_CKPT_NAME}'
    set_seed(0)
    vae = VAE(NUM_FEATURES, CFG["HIDDEN"], CFG["LATENT"]).to(DEVICE)
    opt = torch.optim.Adam(vae.parameters(), lr=CFG["VAE_LR"], weight_decay=CFG["VAE_WD"])
    X = torch.tensor(X_PAN, dtype=torch.float32)
    n, bs = X.shape[0], CFG["VAE_BATCH"]
    print(f"Training VAE: {CFG['VAE_EPOCHS']} epochs, {n} samples, batch {bs}")
    print(f"  checkpointing to {ckpt} every 25 epochs")
    t0 = time.time()
    for ep in range(CFG["VAE_EPOCHS"]):
        vae.train(); perm = torch.randperm(n); tot = 0.0
        for i in range(0, n, bs):
            xb = X[perm[i:i+bs]].to(DEVICE)
            opt.zero_grad(); loss = vae(xb); loss.backward(); opt.step()
            tot += loss.item() * len(xb)
        if ep % 25 == 0 or ep == CFG["VAE_EPOCHS"] - 1:
            # Save as we go: this loop is ~an hour of GPU, far too expensive to
            # lose to a failure anywhere after it.
            torch.save(vae.state_dict(), ckpt)
            print(f"  epoch {ep:3d}  loss {tot/n:.4f}  ({time.time()-t0:.0f}s)  [saved]")
    torch.save(vae.state_dict(), ckpt)
    print(f"VAE trained in {time.time()-t0:.0f}s → {ckpt}")
    return vae


VAE_MODEL = train_vae()

## 4 · Survival models & Cox partial-likelihood loss

`PartialNLL`, risk-set matrix and C-index are ported directly from the repo
(`models.py`, `phase2_reproduction.py`). VAECox here is the **fine-tuned**
variant: pretrained encoder + Coxnnet head, all weights trainable.

In [ ]:
import copy
from lifelines.utils import concordance_index


class PartialNLL(nn.Module):
    def forward(self, theta, R, censored):
        observed = 1 - censored
        num_obs = torch.sum(observed)
        if num_obs == 0:
            return (theta * 0).sum()
        exp_theta = torch.exp(theta)
        return -(torch.sum((theta.reshape(-1) -
                 torch.log(torch.sum(exp_theta * R.t(), 0))) * observed) / num_obs)


class CoxLinear(nn.Module):
    def __init__(self, p):
        super().__init__(); self.fc1 = nn.Linear(p, 1); nn.init.xavier_normal_(self.fc1.weight)
    def forward(self, x): return self.fc1(x)


class Coxnnet(nn.Module):
    def __init__(self, p):
        super().__init__(); h = int(np.ceil(p ** 0.5))
        self.fc1 = nn.Linear(p, h); self.fc2 = nn.Linear(h, 1)
    def forward(self, x): return self.fc2(torch.tanh(self.fc1(x)))


class CoxMLP(nn.Module):
    def __init__(self, p, nhid=100, dropout=0.0):
        super().__init__(); self.fc1 = nn.Linear(p, nhid); self.fc2 = nn.Linear(nhid, 1); self.d = dropout
    def forward(self, x):
        x = F.dropout(F.relu(self.fc1(x)), self.d, training=self.training); return self.fc2(x)


class VAECox(nn.Module):
    """Paper's method: pretrained VAE encoder (FINE-TUNED) + Coxnnet(128).

    The encoder is deep-copied. Assigning `pretrained_vae.encode` directly would
    share the module object, so every fine-tuning run would keep mutating the one
    pretrained VAE in place and each cancer/seed would silently start from the
    previous run's weights.
    """
    def __init__(self, pretrained_vae, latent=128):
        super().__init__()
        vae = copy.deepcopy(pretrained_vae)
        self.encode = vae.encode
        self.encode_mu = vae.encode_mu
        self.cox = Coxnnet(latent)
        for p in self.parameters():
            p.requires_grad = True
    def forward(self, x):
        return self.cox(self.encode_mu(self.encode(x)))


def make_R(y):
    n = len(y); R = np.zeros((n, n), dtype=np.float32)
    for i in range(n): R[i, :] = (y >= y[i])
    return R


def cindex_safe(y, pred, c):
    ev = (c == 0)
    if ev.sum() == 0: return float("nan")
    try: return concordance_index(y, pred, ev)
    except Exception: return float("nan")


def train_eval(model, Xtr, ytr, ctr, Xte, yte, cte, lr, wd, epochs, lasso=0.0):
    model = model.to(DEVICE)
    lossf = PartialNLL()
    X = torch.tensor(Xtr, dtype=torch.float32, device=DEVICE)
    R = torch.tensor(make_R(ytr), dtype=torch.float32, device=DEVICE)
    c = torch.tensor(ctr, dtype=torch.float32, device=DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    model.train()
    for _ in range(epochs):
        opt.zero_grad(); theta = model(X); loss = lossf(theta, R, c)
        if lasso > 0:
            loss = loss + lasso * sum(p.abs().sum() for p in model.fc1.parameters())
        if torch.isnan(loss) or torch.isinf(loss): break
        loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        pred = -model(torch.tensor(Xte, dtype=torch.float32, device=DEVICE)).reshape(-1).cpu().numpy()
    return cindex_safe(yte, pred, cte)

## 5 · Phase 2 — reproduce the headline C-index table

All models on all 10 cancers × 10 seeds. Hyperparameters are searched once per
cancer on a validation split, then applied across seeds: the three neural
models (Coxnnet, CoxMLP, VAECox) over a reduced (lr, weight-decay) grid, and
CoxLasso/CoxRidge — whose penalty *is* essentially the whole model — over a
13-point regularization path instead of sharing the neural grid. Reports
mean ± std → `results/cindex_comparison.csv`, per-seed values →
`results/cindex_by_seed.csv`, and a paired VAECox-vs-baseline comparison (win
count, Wilcoxon signed-rank, bootstrap CI) → `results/paired_stats.csv` —
the marginal seed-to-seed std mixes in split difficulty shared by every model,
so it is not the right quantity for judging whether a per-cancer margin is
"within noise".


In [ ]:
# Neural models keep the original 3-combo (lr, wd) grid — architecture and
# initialisation carry most of their behaviour, so a coarse search is defensible.
NEURAL_HP_GRID = [(1e-3, 1e-5), (1e-3, 1e-3), (1e-4, 1e-5)] if CFG["HP_SEARCH"] else [(1e-3, 1e-5)]

# CoxLasso/CoxRidge are essentially JUST their penalty term, so piggybacking on
# the 3-combo neural grid under-tunes them relative to their conventional usage
# (a regularization path of dozens of values) — flagged in both reviews as
# capable of accounting for VAECox's entire reported mean-C-index advantage.
# lr is held fixed (as before) and only the penalty is searched, over a wide path.
LINEAR_LR = 1e-4
PENALTY_GRID = [1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1, 1.0, 3.0, 10.0] \
    if CFG["HP_SEARCH"] else [1e-3]


def build_model(name, p):
    if name == "CoxLasso":  return CoxLinear(p)
    if name == "CoxRidge":  return CoxLinear(p)
    if name == "Coxnnet":   return Coxnnet(p)
    if name == "CoxMLP":    return CoxMLP(p)
    if name == "VAECox":    return VAECox(VAE_MODEL, CFG["LATENT"])
    raise ValueError(name)


def fit_one(name, Xtr, ytr, ctr, Xte, yte, cte, hp):
    """`hp` is a penalty scalar for CoxLasso/CoxRidge, an (lr, wd) pair otherwise."""
    m = build_model(name, Xtr.shape[1])
    if name == "CoxLasso":
        return train_eval(m, Xtr, ytr, ctr, Xte, yte, cte, LINEAR_LR, 0.0,
                          CFG["SURV_EPOCHS"], lasso=hp)
    if name == "CoxRidge":
        return train_eval(m, Xtr, ytr, ctr, Xte, yte, cte, LINEAR_LR, hp, CFG["SURV_EPOCHS"])
    lr, wd = hp
    return train_eval(m, Xtr, ytr, ctr, Xte, yte, cte, lr, wd, CFG["SURV_EPOCHS"])


def search_hp(name, cohort):
    """Pick the best hyperparameter on the seed-0 val split; returned in the
    form fit_one expects (a penalty scalar for the linear models, an (lr, wd)
    pair for the neural ones)."""
    grid = PENALTY_GRID if name in ("CoxLasso", "CoxRidge") else NEURAL_HP_GRID
    if len(grid) == 1:
        return grid[0]
    Xtr, Xte, ytr, yte, ctr, cte = make_split(cohort, 0)
    best, best_hp = -1, grid[0]
    for hp in grid:
        ci = fit_one(name, Xtr, ytr, ctr, Xte, yte, cte, hp)
        if not np.isnan(ci) and ci > best:
            best, best_hp = ci, hp
    return best_hp


MODELS = ["CoxLasso", "CoxRidge", "Coxnnet", "CoxMLP", "VAECox"]


def run_phase2():
    rows, seed_rows = [], []
    for cohort in CFG["PAPER_10"]:
        if cohort not in COHORT_DFS:
            continue
        print(f"\n── {cohort} ──")
        hp = {m: search_hp(m, cohort) for m in MODELS}
        for m in MODELS:
            vals = []
            for seed in CFG["SEEDS"]:
                set_seed(seed)
                Xtr, Xte, ytr, yte, ctr, cte = make_split(cohort, seed)
                ci = fit_one(m, Xtr, ytr, ctr, Xte, yte, cte, hp[m])
                vals.append(ci)
                seed_rows.append(dict(cancer=cohort, model=m, seed=seed, cindex=ci))
            v = [x for x in vals if not np.isnan(x)]
            mean = np.mean(v) if v else float("nan")
            std  = np.std(v) if v else float("nan")
            rows.append(dict(cancer=cohort, model=m, mean_cindex=round(mean, 4),
                             std_cindex=round(std, 4), n_valid=len(v)))
            print(f"  {m:9s}: {mean:.3f} ± {std:.3f}  (hp={hp[m]}, {len(v)} seeds)")
    df = pd.DataFrame(rows)
    df.to_csv(f'{CFG["OUT"]}/results/cindex_long.csv', index=False)
    seed_df = pd.DataFrame(seed_rows)
    seed_df.to_csv(f'{CFG["OUT"]}/results/cindex_by_seed.csv', index=False)
    # wide table (mean) + wins
    wide = df.pivot(index="model", columns="cancer", values="mean_cindex")
    wide["Mean"] = wide.mean(axis=1)
    wide.to_csv(f'{CFG["OUT"]}/results/cindex_comparison.csv')
    wins = {m: 0 for m in MODELS}
    for cohort in wide.columns[:-1]:
        col = wide[cohort].dropna()
        if len(col): wins[col.idxmax()] += 1
    print("\n=== WINS (VAECox target: 7/10) ===")
    for m, w in sorted(wins.items(), key=lambda x: -x[1]):
        print(f"  {m:9s}: {w}/10")
    return df, seed_df, wide, wins


def phase2_resumable():
    """Phase 2 is the single longest non-restartable stage, so it reloads its own
    long-form CSVs and rebuilds the derived table/wins from them."""
    prev = find_prev("cindex_long.csv") if CFG["RESUME"] else None
    prev_seed = find_prev("cindex_by_seed.csv") if CFG["RESUME"] else None
    if not prev:
        return run_phase2()
    df = pd.read_csv(prev)
    print(f"[resume] cindex_long.csv: {len(df)} rows from {prev} — Phase 2 not re-run")
    dest = f'{CFG["OUT"]}/results/cindex_long.csv'
    if os.path.abspath(prev) != os.path.abspath(dest):
        df.to_csv(dest, index=False)
    if prev_seed:
        seed_df = pd.read_csv(prev_seed)
        dest_seed = f'{CFG["OUT"]}/results/cindex_by_seed.csv'
        if os.path.abspath(prev_seed) != os.path.abspath(dest_seed):
            seed_df.to_csv(dest_seed, index=False)
    else:
        seed_df = pd.DataFrame(columns=["cancer", "model", "seed", "cindex"])
        print("  [resume] no cindex_by_seed.csv found — paired stats will be skipped")
    wide = df.pivot(index="model", columns="cancer", values="mean_cindex")
    wide["Mean"] = wide.mean(axis=1)
    wide.to_csv(f'{CFG["OUT"]}/results/cindex_comparison.csv')
    wins = {m: 0 for m in df.model.unique()}
    for cohort in [c for c in wide.columns if c != "Mean"]:
        col = wide[cohort].dropna()
        if len(col):
            wins[col.idxmax()] += 1
    print("=== WINS (VAECox target: 7/10) ===")
    for m, w in sorted(wins.items(), key=lambda x: -x[1]):
        print(f"  {m:9s}: {w}/10")
    return df, seed_df, wide, wins


def paired_stats(seed_df, baseline_models=("CoxLasso", "CoxRidge", "Coxnnet", "CoxMLP")):
    """Per-seed paired comparison of VAECox against each baseline.

    Both reviews flagged that comparing per-cancer margins to the *marginal*
    seed-to-seed std overstates the noise: since all five models are fit on the
    same 10 splits, most of that 0.02-0.08 spread is shared split difficulty,
    which cancels out in a paired difference. This instead reports, per cancer
    and baseline, the per-seed VAECox-minus-baseline differences: mean, a win
    count, a Wilcoxon signed-rank test, and a percentile bootstrap CI.
    """
    if seed_df is None or not len(seed_df):
        print("paired_stats: no per-seed data available — skipping")
        return pd.DataFrame()
    from scipy.stats import wilcoxon
    rows = []
    wide = seed_df.pivot_table(index=["cancer", "seed"], columns="model", values="cindex")
    for cohort, g in wide.groupby(level=0):
        g = g.droplevel(0)
        if "VAECox" not in g.columns:
            continue
        for base in baseline_models:
            if base not in g.columns:
                continue
            paired = g[["VAECox", base]].dropna()
            if len(paired) < 3:
                continue
            diff = (paired["VAECox"] - paired[base]).values
            n = len(diff)
            wins = int((diff > 0).sum())
            rng = np.random.default_rng(42)
            boot = np.array([rng.choice(diff, n, replace=True).mean() for _ in range(2000)])
            lo, hi = np.percentile(boot, [2.5, 97.5])
            try:
                _, p = wilcoxon(diff) if np.any(diff != 0) else (np.nan, 1.0)
            except ValueError:
                p = 1.0
            rows.append(dict(cancer=cohort, baseline=base, n_seeds=n,
                             mean_diff=round(float(np.mean(diff)), 4),
                             wins_vaecox=wins, losses=n - wins,
                             ci95_lo=round(float(lo), 4), ci95_hi=round(float(hi), 4),
                             wilcoxon_p=round(float(p), 4)))
    df = pd.DataFrame(rows)
    df.to_csv(f'{CFG["OUT"]}/results/paired_stats.csv', index=False)
    if len(df):
        sig = df[df.wilcoxon_p < 0.05]
        print(f"paired_stats: {len(df)} (cancer, baseline) pairs; "
              f"{len(sig)} significant at p<0.05 (Wilcoxon signed-rank), "
              f"{int((df.mean_diff > 0).sum())} with VAECox ahead on average")
    return df


PH2_LONG, PH2_SEED, PH2_WIDE, PH2_WINS = phase2_resumable()
print("\n", PH2_WIDE.round(3))
PAIRED_STATS = paired_stats(PH2_SEED)
print(PAIRED_STATS)


## 6 · Phase 3 — extensions

Robustness (missing + noise), fairness (event-rate vs C-index), lightweight
models (latent/hidden sweep), feature importance, Kaplan–Meier. All on the
real data. Robustness (§6j.2) scores every model — including a VAECox-Random
ablation (same architecture, untrained encoder) — rather than CoxRidge vs
VAECox alone, so the bottleneck architecture's contribution can be told apart
from pretraining's.


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

RES = f'{CFG["OUT"]}/results'


# --- 6a robustness: missing features + gaussian noise -----------------------
def robustness(cohort="STAD"):
    out = []
    for frac in [0.0, 0.1, 0.25, 0.5]:
        cis = {"CoxRidge": [], "VAECox": []}
        for seed in CFG["SEEDS"]:
            set_seed(seed)
            Xtr, Xte, ytr, yte, ctr, cte = make_split(cohort, seed)
            rng = np.random.default_rng(seed + 1000)
            mask = rng.random(Xte.shape) >= frac
            Xte_m = Xte * mask
            cis["CoxRidge"].append(fit_one("CoxRidge", Xtr, ytr, ctr, Xte_m, yte, cte, 1e-3))
            cis["VAECox"].append(fit_one("VAECox", Xtr, ytr, ctr, Xte_m, yte, cte, (1e-3, 1e-5)))
        out.append(dict(experiment="missing", level=f"{int(frac*100)}%",
                        CoxRidge=np.nanmean(cis["CoxRidge"]), VAECox=np.nanmean(cis["VAECox"])))
    for sig in [0.0, 0.5, 1.0, 2.0]:
        cis = {"CoxRidge": [], "VAECox": []}
        for seed in CFG["SEEDS"]:
            set_seed(seed)
            Xtr, Xte, ytr, yte, ctr, cte = make_split(cohort, seed)
            rng = np.random.default_rng(seed + 2000)
            Xte_n = Xte + rng.normal(0, sig, Xte.shape).astype(np.float32)
            cis["CoxRidge"].append(fit_one("CoxRidge", Xtr, ytr, ctr, Xte_n, yte, cte, 1e-3))
            cis["VAECox"].append(fit_one("VAECox", Xtr, ytr, ctr, Xte_n, yte, cte, (1e-3, 1e-5)))
        out.append(dict(experiment="noise", level=f"sigma={sig}",
                        CoxRidge=np.nanmean(cis["CoxRidge"]), VAECox=np.nanmean(cis["VAECox"])))
    df = pd.DataFrame(out); df.to_csv(f"{RES}/robustness.csv", index=False)
    print(df.round(3)); return df


# --- 6b fairness: does C-index track #events / cohort size? ------------------
def fairness():
    rows = []
    for cohort in CFG["PAPER_10"]:
        if cohort not in COHORT_DFS: continue
        sub = PH2_LONG[(PH2_LONG.cancer == cohort) & (PH2_LONG.model == "VAECox")]
        if len(sub) == 0: continue
        d = COHORT_DFS[cohort]
        rows.append(dict(cancer=cohort, n=len(d), n_events=int((d.censored == 0).sum()),
                         vaecox_cindex=float(sub.mean_cindex.iloc[0])))
    df = pd.DataFrame(rows); df.to_csv(f"{RES}/fairness.csv", index=False)
    if len(df) > 2:
        r = np.corrcoef(df.n_events, df.vaecox_cindex)[0, 1]
        print(f"corr(events, VAECox C-index) = {r:.3f}")
    return df


# --- 6c lightweight: latent/hidden dimension sweep --------------------------
def lightweight(cohort="STAD"):
    rows = []
    for hidden, latent in [(512, 128), (1024, 128), (4096, 32), (4096, 64), (4096, 128)]:
        set_seed(0)
        vae = VAE(NUM_FEATURES, hidden, latent).to(DEVICE)
        opt = torch.optim.Adam(vae.parameters(), lr=CFG["VAE_LR"], weight_decay=CFG["VAE_WD"])
        X = torch.tensor(X_PAN, dtype=torch.float32); n, bs = X.shape[0], CFG["VAE_BATCH"]
        t0 = time.time()
        for ep in range(min(100, CFG["VAE_EPOCHS"])):
            perm = torch.randperm(n)
            for i in range(0, n, bs):
                xb = X[perm[i:i+bs]].to(DEVICE)
                opt.zero_grad(); vae(xb).backward(); opt.step()
        train_sec = time.time() - t0
        cis = []
        for seed in CFG["SEEDS"]:
            set_seed(seed)
            Xtr, Xte, ytr, yte, ctr, cte = make_split(cohort, seed)
            m = VAECox(vae, latent)
            cis.append(train_eval(m, Xtr, ytr, ctr, Xte, yte, cte, 1e-3, 1e-5, CFG["SURV_EPOCHS"]))
        n_params = sum(p.numel() for p in vae.parameters())
        rows.append(dict(hidden=hidden, latent=latent, n_params=n_params,
                         train_sec=round(train_sec, 1), mean_cindex=round(np.nanmean(cis), 4)))
        print(rows[-1])
    df = pd.DataFrame(rows); df.to_csv(f"{RES}/lightweight.csv", index=False); return df


# --- 6d feature importance: |Cox weights| on Ridge --------------------------
def feature_importance(cohort="STAD", topk=25):
    set_seed(0)
    Xtr, Xte, ytr, yte, ctr, cte = make_split(cohort, 0)
    m = CoxLinear(Xtr.shape[1]).to(DEVICE)
    train_eval(m, Xtr, ytr, ctr, Xte, yte, cte, 1e-4, 1e-3, CFG["SURV_EPOCHS"])
    w = m.fc1.weight.detach().cpu().numpy().reshape(-1)
    idx = np.argsort(-np.abs(w))[:topk]
    df = pd.DataFrame(dict(gene=[GENES[i] for i in idx], weight=w[idx].round(4)))
    df.to_csv(f"{RES}/feature_importance.csv", index=False)
    print(df.head(10)); return df


# --- 6e Kaplan-Meier by predicted risk --------------------------------------
def kaplan_meier(cohort="STAD"):
    set_seed(0)
    Xtr, Xte, ytr, yte, ctr, cte = make_split(cohort, 0)
    m = VAECox(VAE_MODEL, CFG["LATENT"]).to(DEVICE)
    train_eval(m, Xtr, ytr, ctr, Xte, yte, cte, 1e-3, 1e-5, CFG["SURV_EPOCHS"])
    m.eval()
    with torch.no_grad():
        risk = m(torch.tensor(Xte, dtype=torch.float32, device=DEVICE)).reshape(-1).cpu().numpy()
    hi = risk >= np.median(risk)
    ev = (cte == 0)
    fig, ax = plt.subplots(figsize=(6, 4))
    kmf = KaplanMeierFitter()
    for grp, lab in [(hi, "High risk"), (~hi, "Low risk")]:
        if grp.sum() > 0:
            kmf.fit(yte[grp], ev[grp], label=lab); kmf.plot_survival_function(ax=ax)
    lr = logrank_test(yte[hi], yte[~hi], ev[hi], ev[~hi])
    ax.set_title(f"{cohort} — KM by VAECox risk (log-rank p={lr.p_value:.3f})")
    ax.set_xlabel("Days"); ax.set_ylabel("Survival probability")
    fig.tight_layout(); fig.savefig(f'{CFG["OUT"]}/figures/km_{cohort}.png', dpi=120)
    print(f"{cohort} log-rank p = {lr.p_value:.4f}")
    return lr.p_value


# §6a and §6c cover one cohort each; §6j.2 and §6i.a redo both across every
# cohort, so running them here duplicates ~40-60 min of GPU time for results
# that get superseded. Set CFG["SKIP_SUPERSEDED"]=False to run them anyway.
if CFG["SKIP_SUPERSEDED"]:
    print("### 6a robustness  — SKIPPED (superseded by §6j.2, all cohorts)")
    print("### 6c lightweight — SKIPPED (superseded by §6i.a, all cohorts)")
    ROB = pd.DataFrame()
    LIGHT = pd.DataFrame()
else:
    print("\n### 6a robustness"); ROB = robustness()
    print("\n### 6c lightweight"); LIGHT = lightweight()

FAIR, FI = pd.DataFrame(), pd.DataFrame()
try:
    print("\n### 6b fairness");   FAIR = fairness()
    print("\n### 6d importance"); FI = feature_importance()
    print("\n### 6e Kaplan-Meier")
    for c in ["STAD", "BLCA", "KIRC"]:
        if c in COHORT_DFS: kaplan_meier(c)
except Exception:
    import traceback
    print("  !! §6b/6d/6e FAILED — continuing"); traceback.print_exc()

## 6g · Extension helpers + clinical metadata

`train_eval` only ever returns a C-index, but the fairness, robustness,
interpretability and Kaplan–Meier extensions all need the **risk scores**
themselves and the identity of the test patients. This cell adds a reusable
fit/predict pair, an index-returning version of the split, and re-reads the
clinical columns (`age`, `gender`, `stage`, `histological_type`) that §1 dropped
when it kept only survival.

In [ ]:
# ---------------------------------------------------------------------------
# 6g · Shared helpers for the extensions below
#      (a) reusable fit / predict that returns RISK SCORES, not just a C-index
#      (b) clinical metadata (age, sex, stage, subtype) for subgroup analysis
# ---------------------------------------------------------------------------

# Fixed hyperparameters for the extension experiments. The §5 HP search is per
# cancer and per model; re-running it inside every extension would multiply the
# runtime for no scientific gain, so the extensions use one sensible setting and
# say so in the card.
DEFAULT_HP = {"CoxLasso": (1e-4, 0.0), "CoxRidge": (1e-4, 1e-3), "Coxnnet": (1e-3, 1e-5),
              "CoxMLP": (1e-3, 1e-5), "VAECox": (1e-3, 1e-5),
              # Same architecture as VAECox, but the encoder is never pretrained —
              # isolates "deep net with a 128-d bottleneck" from "pretrained on
              # pan-cancer data" in the robustness comparison (6j.2).
              "VAECox-Random": (1e-3, 1e-5)}


def _fit(model, Xtr, ytr, ctr, lr, wd, epochs, lasso=0.0, device=None):
    """train_eval() without the evaluation — returns the fitted model."""
    device = device or DEVICE
    model = model.to(device)
    lossf = PartialNLL()
    X = torch.tensor(Xtr, dtype=torch.float32, device=device)
    R = torch.tensor(make_R(ytr), dtype=torch.float32, device=device)
    c = torch.tensor(ctr, dtype=torch.float32, device=device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    model.train()
    for _ in range(epochs):
        opt.zero_grad()
        loss = lossf(model(X), R, c)
        if lasso > 0:
            loss = loss + lasso * sum(p.abs().sum() for p in model.fc1.parameters())
        if torch.isnan(loss) or torch.isinf(loss):
            break
        loss.backward(); opt.step()
    model.eval()
    return model


def _risk(model, X, device=None):
    """Risk score, sign-matched to train_eval (higher = worse prognosis)."""
    device = device or DEVICE
    with torch.no_grad():
        return -model(torch.tensor(X, dtype=torch.float32, device=device)).reshape(-1).cpu().numpy()


def fit_risk(name, Xtr, ytr, ctr, Xte, lr=None, wd=None, epochs=None,
             device=None, vae=None, latent=None):
    """Train `name` and return (model, risk scores on Xte)."""
    lr0, wd0 = DEFAULT_HP[name]
    lr = lr0 if lr is None else lr
    wd = wd0 if wd is None else wd
    epochs = epochs or CFG["SURV_EPOCHS"]
    latent = latent or CFG["LATENT"]
    if name == "VAECox":
        m = VAECox(vae if vae is not None else VAE_MODEL, latent)
    elif name == "VAECox-Random":
        m = VAECox(VAE(NUM_FEATURES, CFG["HIDDEN"], latent), latent)
    else:
        m = build_model(name, Xtr.shape[1])
    m = _fit(m, Xtr, ytr, ctr, lr, wd, epochs, 0.01 if name == "CoxLasso" else 0.0, device)
    return m, _risk(m, Xte, device)


def split_indices(cohort, seed):
    """Positional train/test indices for the exact same split make_split() builds.
    Needed because the extensions have to map test rows back to patients."""
    X, y, c = cohort_matrix(cohort)
    try:
        strata = pd.qcut(y, q=min(5, len(np.unique(y))), labels=False, duplicates="drop")
    except Exception:
        strata = c
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    return next(sss.split(X, strata))


def split_arrays(cohort, tr, te, cols=None):
    """Z-normalised train/test arrays for given indices (scaler fit on train only).
    `cols` optionally restricts to a gene subset."""
    X, y, c = cohort_matrix(cohort)
    if cols is not None:
        X = X[:, cols]
    sc = StandardScaler().fit(X[tr])
    return (sc.transform(X[tr]).astype(np.float32), sc.transform(X[te]).astype(np.float32),
            y[tr], y[te], c[tr], c[te])


# ---- clinical metadata ------------------------------------------------------
SUBGROUP_VARS = ["age_group", "sex", "stage", "subtype"]
MIN_SUBTYPE_N = 25          # rarer histologies are collapsed to NaN and skipped


def _is_freetext(label):
    """TCGA histology free-text placeholders — 'Other, specify', 'Mixed Histology
    (please specify)'. Whoever filled the form declined to name a subtype, so the
    patients in such a bucket share nothing clinically and a C-index gap between
    two of them is not a disparity. ('NOS' is kept: a genuine, if broad,
    pathology category.)"""
    return "specify" in str(label).lower()


def _first_col(clin, names):
    C = {c.lower(): c for c in clin.columns}
    for n in names:
        if n in C:
            return clin[C[n]]
    return None


def _age_group(v):
    a = pd.to_numeric(v, errors="coerce")
    return pd.cut(a, [0, 50, 65, 200], labels=["<=50", "51-65", ">65"]).astype(object)


def _stage_group(v):
    """TCGA stage strings ('Stage IIIA', 'Stage IV') → Early / Late. Cohorts graded
    rather than staged (e.g. LGG) simply yield NaN and drop out of the analysis."""
    s = v.astype(str).str.upper().str.replace("STAGE", "", regex=False).str.strip()
    out = pd.Series(np.nan, index=v.index, dtype=object)
    out[s.str.match(r"^(I|II)[ABC]?$", na=False)] = "Early (I-II)"
    out[s.str.match(r"^(III|IV)[ABC]?$", na=False)] = "Late (III-IV)"
    return out


def load_clinical_subgroups():
    """Re-read each cohort's clinicalMatrix for the columns §1 dropped, aligned to
    the sample order of COHORT_DFS."""
    out = {}
    for folder in sorted(glob.glob(os.path.join(TCGA_ROOT, "*"))):
        if not os.path.isdir(folder):
            continue
        files = os.listdir(folder)
        cli_f = next((f for f in files if "clinicalMatrix" in f), None)
        exp_f = next((f for f in files if "HiSeqV2" in f), None)
        if not cli_f or not exp_f:
            continue
        code = cohort_code(folder, exp_f)
        if code not in COHORT_DFS:
            continue
        clin = _read_tsv(os.path.join(folder, cli_f))
        clin.index = clin.index.astype(str).str[:15]
        clin = clin[~clin.index.duplicated(keep="first")]

        sub = pd.DataFrame(index=clin.index)
        age = _first_col(clin, ["age_at_initial_pathologic_diagnosis", "age_at_diagnosis", "age"])
        sub["age_group"] = _age_group(age) if age is not None else np.nan
        sex = _first_col(clin, ["gender", "sex"])
        sub["sex"] = (sex.astype(str).str.upper().replace({"NAN": np.nan, "": np.nan})
                      if sex is not None else np.nan)
        stg = _first_col(clin, ["pathologic_stage", "clinical_stage", "tumor_stage"])
        sub["stage"] = _stage_group(stg) if stg is not None else np.nan
        hist = _first_col(clin, ["histological_type", "histology"])
        if hist is not None:
            h = hist.astype(str).replace({"nan": np.nan, "": np.nan})
            keep = h.value_counts()
            keep = set(keep[keep >= MIN_SUBTYPE_N].index)
            sub["subtype"] = h.where(h.isin(keep))
        else:
            sub["subtype"] = np.nan
        out[code] = sub.reindex(COHORT_DFS[code].index)
    return out


CLIN_SUB = load_clinical_subgroups()

print(f"Clinical metadata parsed for {len(CLIN_SUB)}/{len(COHORT_DFS)} cohorts")
print(f"{'cohort':8s}" + "".join(f"{v:>14s}" for v in SUBGROUP_VARS))
for c in CFG["PAPER_10"]:
    s = CLIN_SUB.get(c)
    if s is None:
        print(f"{c:8s}" + "  (no clinicalMatrix)"); continue
    cells = []
    for v in SUBGROUP_VARS:
        col = s[v].dropna()
        cells.append(f"{col.nunique()}g/{len(col)}n" if len(col) else "—")
    print(f"{c:8s}" + "".join(f"{x:>14s}" for x in cells))
print("\n('3g/412n' = 3 distinct groups covering 412 patients; '—' = variable "
      "absent for that cohort, e.g. stage is undefined for LGG)")


## 6h · Subgroup fairness (age · sex · stage · histological subtype)

§6b only asked whether cohort-level C-index tracks cohort size. The roadmap also
asks for performance **by clinical subgroup**, which needs the metadata that §1
discarded. Below: within-stratum C-index per cancer and model, the best–worst
gap per stratum, and cohort-level fairness for *every* model rather than VAECox
alone.

Strata with fewer than 10 test patients or 3 uncensored events are dropped — the
C-index is not estimable there, and reporting it would manufacture disparities
out of noise.

In [ ]:
# ---------------------------------------------------------------------------
# 6h · Subgroup fairness: C-index within age / sex / stage / subtype strata
# ---------------------------------------------------------------------------
MIN_GROUP_N = 10      # patients in a test-set stratum
MIN_GROUP_EV = 3      # uncensored events in that stratum (C-index is undefined below this)


def subgroup_cindex(models=("CoxRidge", "VAECox"), cohorts=None, seeds=None):
    """Per-seed C-index computed *within* each clinical stratum of the test set.

    Risk scores are only comparable inside one fitted model, so the C-index is
    computed per seed and then averaged — pooling raw scores across seeds would
    mix incomparable scales.
    """
    cohorts = cohorts or CFG["PAPER_10"]
    seeds = seeds if seeds is not None else CFG["SEEDS"]
    rows = []
    for cohort in cohorts:
        sub = CLIN_SUB.get(cohort)
        if sub is None or not len(sub.columns):
            continue
        for name in models:
            acc = {}
            for seed in seeds:
                set_seed(seed)
                tr, te = split_indices(cohort, seed)
                Xtr, Xte, ytr, yte, ctr, cte = split_arrays(cohort, tr, te)
                _, risk = fit_risk(name, Xtr, ytr, ctr, Xte)
                lab = sub.iloc[te]
                for var in SUBGROUP_VARS:
                    if var not in lab.columns:
                        continue
                    vals = lab[var]
                    for lev in pd.unique(vals.dropna()):
                        if _is_freetext(lev):
                            continue
                        mask = (vals == lev).values
                        if mask.sum() < MIN_GROUP_N or (cte[mask] == 0).sum() < MIN_GROUP_EV:
                            continue
                        ci = cindex_safe(yte[mask], risk[mask], cte[mask])
                        if not np.isnan(ci):
                            acc.setdefault((var, str(lev)), []).append((ci, int(mask.sum())))
            for (var, lev), v in acc.items():
                cis = [x[0] for x in v]
                rows.append(dict(cancer=cohort, model=name, variable=var, group=lev,
                                 n_seeds=len(cis), mean_n=int(np.mean([x[1] for x in v])),
                                 mean_cindex=round(float(np.mean(cis)), 4),
                                 std_cindex=round(float(np.std(cis)), 4)))
        print(f"  {cohort} done ({sum(r['cancer'] == cohort for r in rows)} strata)")
    df = pd.DataFrame(rows)
    df.to_csv(f"{RES}/subgroup_cindex.csv", index=False)
    # Record what the free-text filter removed, so the exclusion is auditable
    # rather than an invisible judgement call.
    dropped = []
    for cohort in cohorts:
        sub = CLIN_SUB.get(cohort)
        if sub is None or "subtype" not in sub.columns:
            continue
        for lev in pd.unique(sub["subtype"].dropna()):
            if _is_freetext(lev):
                dropped.append(dict(cancer=cohort, variable="subtype", group=str(lev),
                                    n_patients=int((sub["subtype"] == lev).sum()),
                                    reason="TCGA free-text placeholder"))
    pd.DataFrame(dropped).to_csv(f"{RES}/subgroup_excluded_strata.csv", index=False)
    print(f"  excluded {len(dropped)} free-text histology strata "
          f"(see subgroup_excluded_strata.csv)")
    return df


def subgroup_gaps(df):
    """Best-group minus worst-group C-index — the disparity actually worth reporting."""
    if not len(df):
        return pd.DataFrame()
    rows = []
    for (cancer, model, var), g in df.groupby(["cancer", "model", "variable"]):
        if len(g) < 2:
            continue
        hi, lo = g.loc[g.mean_cindex.idxmax()], g.loc[g.mean_cindex.idxmin()]
        rows.append(dict(cancer=cancer, model=model, variable=var,
                         best_group=hi.group, best_cindex=hi.mean_cindex,
                         worst_group=lo.group, worst_cindex=lo.mean_cindex,
                         gap=round(float(hi.mean_cindex - lo.mean_cindex), 4),
                         n_groups=len(g)))
    out = pd.DataFrame(rows).sort_values("gap", ascending=False)
    out.to_csv(f"{RES}/subgroup_gaps.csv", index=False)
    if len(out):
        print("\n  Largest disparities:")
        print(out.head(10).to_string(index=False))
        print("\n  Mean gap by variable:")
        print(out.groupby(["variable", "model"]).gap.mean().round(4).to_string())
    return out


# --- cohort-level fairness, both models (§6b only did VAECox) ---------------
def cohort_fairness():
    rows = []
    for cohort in CFG["PAPER_10"]:
        d = COHORT_DFS[cohort]
        for _, r in PH2_LONG[PH2_LONG.cancer == cohort].iterrows():
            rows.append(dict(cancer=cohort, model=r.model, n_patients=len(d),
                             n_events=int((d.censored == 0).sum()),
                             event_rate=round(float((d.censored == 0).mean()), 4),
                             mean_cindex=r.mean_cindex))
    df = pd.DataFrame(rows)
    df.to_csv(f"{RES}/cohort_fairness.csv", index=False)
    print("\n  corr(#events, C-index) per model:")
    for m, g in df.groupby("model"):
        g = g.dropna(subset=["mean_cindex"])
        if len(g) > 2:
            print(f"    {m:9s} events r={np.corrcoef(g.n_events, g.mean_cindex)[0,1]:+.3f}   "
                  f"cohort-size r={np.corrcoef(g.n_patients, g.mean_cindex)[0,1]:+.3f}")
    return df


print("### 6h.1 clinical subgroups")
SUBGROUP = cached("subgroup_cindex", subgroup_cindex)
GAPS = subgroup_gaps(SUBGROUP)
print("\n### 6h.2 cohort-level fairness")
COHORT_FAIR = cached("cohort_fairness", cohort_fairness)

## 6i · Low-resource extensions

§6c swept VAE sizes on one cohort and reported a single mean. The roadmap asks
two sharper questions:

* **Equity of compression** — if a smaller VAE loses accuracy, does it lose *more*
  on the cohorts that already have the fewest uncensored events? A cheap model
  that only stays accurate on large, well-studied cancers is not accessible.
* **Feature budget & CPU feasibility** — how few genes can be kept before the
  C-index collapses, and can the whole pipeline actually run on a CPU? Gene
  selection here is by pan-cancer **variance only**, which never looks at survival
  labels, so the train/test split stays clean.

In [ ]:
# ---------------------------------------------------------------------------
# 6i · Low-resource extensions
#      (a) does shrinking the VAE hurt small cohorts more than large ones?
#      (b) how few genes can you keep and still predict — and does it fit on CPU?
# ---------------------------------------------------------------------------
LIGHT_CONFIGS = [("h4096_l128", 4096, 128),   # paper size (reference point)
                 ("h1024_l64",  1024,  64),
                 ("h512_l32",    512,  32),
                 ("h256_l16",    256,  16)]
SWEEP_VAE_EPOCHS = min(100, CFG["VAE_EPOCHS"])   # equal budget for every config


def train_vae_dims(hidden, latent, cols=None, epochs=None, device=None, seed=0):
    """Pretrain a VAE of the given size on pan-cancer expression (optionally on a
    gene subset). Returns (vae, seconds, n_params)."""
    device = device or DEVICE
    epochs = epochs or SWEEP_VAE_EPOCHS
    Xp = X_PAN if cols is None else X_PAN[:, cols]
    set_seed(seed)
    vae = VAE(Xp.shape[1], hidden, latent).to(device)
    opt = torch.optim.Adam(vae.parameters(), lr=CFG["VAE_LR"], weight_decay=CFG["VAE_WD"])
    X = torch.tensor(Xp, dtype=torch.float32)
    n, bs = X.shape[0], CFG["VAE_BATCH"]
    t0 = time.time()
    for _ in range(epochs):
        vae.train()
        perm = torch.randperm(n)
        for i in range(0, n, bs):
            xb = X[perm[i:i + bs]].to(device)
            opt.zero_grad(); vae(xb).backward(); opt.step()
    vae.eval()
    return vae, round(time.time() - t0, 1), sum(p.numel() for p in vae.parameters())


# --- 6i.a lightweight models, evaluated on EVERY cohort ---------------------
def lightweight_by_cancer(cohorts=None, seeds=None):
    cohorts = cohorts or CFG["PAPER_10"]
    seeds = seeds if seeds is not None else CFG["SEEDS"][:5]
    rows = []
    for cfg_name, hidden, latent in LIGHT_CONFIGS:
        vae, secs, n_params = train_vae_dims(hidden, latent)
        print(f"  {cfg_name}: {n_params/1e6:.1f}M params, pretrained in {secs}s")
        for cohort in cohorts:
            cis = []
            for seed in seeds:
                set_seed(seed)
                tr, te = split_indices(cohort, seed)
                Xtr, Xte, ytr, yte, ctr, cte = split_arrays(cohort, tr, te)
                _, risk = fit_risk("VAECox", Xtr, ytr, ctr, Xte, vae=vae, latent=latent)
                cis.append(cindex_safe(yte, risk, cte))
            rows.append(dict(config=cfg_name, hidden=hidden, latent=latent,
                             n_params=n_params, train_sec=secs, cancer=cohort,
                             cindex=round(float(np.nanmean(cis)), 4),
                             n_events=int((COHORT_DFS[cohort].censored == 0).sum()),
                             n_patients=int(len(COHORT_DFS[cohort]))))
        del vae; gc.collect()
        if DEVICE.type == "cuda": torch.cuda.empty_cache()
    df = pd.DataFrame(rows)
    df.to_csv(f"{RES}/lightweight_by_cancer.csv", index=False)
    print("\n  mean C-index per config:")
    print(df.groupby("config").cindex.mean().round(4).to_string())
    return df


def lightweight_disparity(light_df):
    """Does compressing the model cost *more* C-index on small / low-event
    cohorts? That is the equity question the roadmap asks: a cheap model that is
    only cheap for well-resourced cancers is not actually accessible."""
    full = LIGHT_CONFIGS[0][0]
    base = light_df[light_df.config == full].set_index("cancer").cindex
    sub = light_df[light_df.config != full].copy()
    sub["delta_vs_full"] = (sub.cindex - sub.cancer.map(base)).round(4)
    sub.to_csv(f"{RES}/lightweight_disparity.csv", index=False)
    med = light_df.groupby("cancer").n_events.first().median()
    print(f"\n  median events across cohorts = {med:.0f}; splitting there:")
    for cfg_name, g in sub.groupby("config"):
        small = g[g.n_events <= med].delta_vs_full.mean()
        large = g[g.n_events > med].delta_vs_full.mean()
        r = (np.corrcoef(g.n_events, g.delta_vs_full)[0, 1]
             if g.delta_vs_full.notna().sum() > 2 else float("nan"))
        print(f"    {cfg_name:11s} Δ small-cohort {small:+.4f} | "
              f"Δ large-cohort {large:+.4f} | corr(events, Δ) = {r:+.3f}")
    return sub


# --- 6i.b feature budget + CPU feasibility ---------------------------------
def top_variance_genes(k):
    """Unsupervised (survival labels never touched) → no leakage into the split."""
    return np.argsort(-X_PAN.var(axis=0))[:k]


def feature_subset_accessibility(cohorts=None, seeds=None,
                                 k_list=(100, 500, 1000, 5000, None),
                                 cpu_check_k=1000):
    cohorts = cohorts or CFG["PAPER_10"][:3]
    seeds = seeds if seeds is not None else CFG["SEEDS"][:5]
    rows = []
    for k in k_list:
        cols = None if k is None else top_variance_genes(k)
        kk = NUM_FEATURES if k is None else k
        vae, vae_secs, n_params = train_vae_dims(512, 32, cols=cols)
        per_model = {"CoxRidge": [], "VAECox": []}
        t0 = time.time()
        for cohort in cohorts:
            for seed in seeds:
                set_seed(seed)
                tr, te = split_indices(cohort, seed)
                Xtr, Xte, ytr, yte, ctr, cte = split_arrays(cohort, tr, te, cols=cols)
                _, r_ridge = fit_risk("CoxRidge", Xtr, ytr, ctr, Xte)
                _, r_vae = fit_risk("VAECox", Xtr, ytr, ctr, Xte, vae=vae, latent=32)
                per_model["CoxRidge"].append(cindex_safe(yte, r_ridge, cte))
                per_model["VAECox"].append(cindex_safe(yte, r_vae, cte))
        surv_secs = round(time.time() - t0, 1)
        for name, v in per_model.items():
            rows.append(dict(k_genes=kk, model=name,
                             mean_cindex=round(float(np.nanmean(v)), 4),
                             std_cindex=round(float(np.nanstd(v)), 4),
                             n_fits=int(np.sum(~np.isnan(v))),
                             vae_params=n_params, vae_pretrain_sec=vae_secs,
                             survival_fit_sec=surv_secs, device=str(DEVICE)))
        print(f"  k={kk:>6}: VAE {vae_secs}s | survival {surv_secs}s | "
              + " ".join(f"{n} {np.nanmean(v):.3f}" for n, v in per_model.items()))
        del vae; gc.collect()
        if DEVICE.type == "cuda": torch.cuda.empty_cache()

    # Same config, forced onto CPU — the actual "can a student run this?" test.
    if cpu_check_k:
        cpu = torch.device("cpu")
        cols = top_variance_genes(cpu_check_k)
        vae, vae_secs, n_params = train_vae_dims(512, 32, cols=cols, epochs=min(20, SWEEP_VAE_EPOCHS),
                                                 device=cpu)
        cohort = cohorts[0]
        t0, cis = time.time(), []
        for seed in seeds[:3]:
            set_seed(seed)
            tr, te = split_indices(cohort, seed)
            Xtr, Xte, ytr, yte, ctr, cte = split_arrays(cohort, tr, te, cols=cols)
            _, risk = fit_risk("VAECox", Xtr, ytr, ctr, Xte, vae=vae, latent=32, device=cpu)
            cis.append(cindex_safe(yte, risk, cte))
        rows.append(dict(k_genes=cpu_check_k, model="VAECox (CPU-only)",
                         mean_cindex=round(float(np.nanmean(cis)), 4),
                         std_cindex=round(float(np.nanstd(cis)), 4),
                         n_fits=len(cis), vae_params=n_params,
                         vae_pretrain_sec=vae_secs,
                         survival_fit_sec=round(time.time() - t0, 1), device="cpu"))
        print(f"  CPU-only check (k={cpu_check_k}, {min(20, SWEEP_VAE_EPOCHS)} VAE epochs): "
              f"{vae_secs}s pretrain + {rows[-1]['survival_fit_sec']}s for 3 fits on {cohort}, "
              f"C-index {rows[-1]['mean_cindex']}")
        del vae; gc.collect()

    df = pd.DataFrame(rows)
    df.to_csv(f"{RES}/feature_subset.csv", index=False)
    return df


print("### 6i.a lightweight models across all cohorts")
LIGHT_CANCER = cached("lightweight_by_cancer", lightweight_by_cancer)
LIGHT_GAP = lightweight_disparity(LIGHT_CANCER)
print("\n### 6i.b feature budget + CPU feasibility")
FEATSUB = cached("feature_subset", feature_subset_accessibility)

## 6j · Interpretability, full-cohort robustness, KM everywhere

Four things §6 only did partially:

* **Permutation importance** — a model-agnostic, SHAP-style attribution. VAECox's
  weights are uninterpretable per gene (the encoder mixes all of them), so
  `|Cox weight|` cannot rank genes for it; shuffling one gene's column and
  measuring the C-index drop can.
* **Robustness on all 10 cohorts**, not just STAD — scored for all 5 models plus
  a VAECox-Random ablation (identical architecture, encoder never pretrained),
  not just CoxRidge vs VAECox, and each model is trained once per (cohort,
  seed) then scored under every corruption level, instead of refitting per
  level.
* **Kaplan–Meier for every cohort**, with log-rank p-values collected in one table.


In [ ]:
# ---------------------------------------------------------------------------
# 6j · Interpretability, robustness across every cohort, KM everywhere,
#      and a direct cell-by-cell comparison against the paper's Table 1.
# ---------------------------------------------------------------------------

# --- 6j.1 permutation importance (model-agnostic, SHAP-style) ---------------
def permutation_importance(cohort, name="VAECox", n_candidates=300,
                           n_repeats=3, top_k=25, seed=0):
    """Drop in test C-index when a gene's values are shuffled across patients.

    Model-agnostic, so VAECox (whose weights say nothing about single genes,
    because the encoder mixes all of them) and CoxRidge are on equal footing.
    Permuting all ~20k genes is wasteful, so a cheap CoxRidge |weight| screen
    picks `n_candidates` genes and only those are permuted.
    """
    set_seed(seed)
    tr, te = split_indices(cohort, seed)
    Xtr, Xte, ytr, yte, ctr, cte = split_arrays(cohort, tr, te)

    screen = _fit(CoxLinear(Xtr.shape[1]), Xtr, ytr, ctr, 1e-4, 1e-3, CFG["SURV_EPOCHS"])
    w = screen.fc1.weight.detach().cpu().numpy().reshape(-1)
    cand = np.argsort(-np.abs(w))[:n_candidates]

    model, risk = fit_risk(name, Xtr, ytr, ctr, Xte)
    base = cindex_safe(yte, risk, cte)
    if np.isnan(base):
        return pd.DataFrame()

    rng = np.random.default_rng(seed + 3000)
    rows = []
    for j in cand:
        drops = []
        for _ in range(n_repeats):
            Xp = Xte.copy()
            Xp[:, j] = Xp[rng.permutation(len(Xp)), j]
            ci = cindex_safe(yte, _risk(model, Xp), cte)
            if not np.isnan(ci):
                drops.append(base - ci)
        if drops:
            rows.append(dict(cancer=cohort, model=name, gene=GENES[j],
                             base_cindex=round(float(base), 4),
                             mean_drop=round(float(np.mean(drops)), 5),
                             std_drop=round(float(np.std(drops)), 5)))
    df = pd.DataFrame(rows).sort_values("mean_drop", ascending=False).head(top_k)
    df.insert(2, "rank", range(1, len(df) + 1))
    return df


def run_importance(cohorts=None, models=("CoxRidge", "VAECox")):
    cohorts = cohorts or CFG["PAPER_10"][:3]
    out = [permutation_importance(c, m) for c in cohorts for m in models]
    out = [d for d in out if len(d)]
    df = pd.concat(out, ignore_index=True) if out else pd.DataFrame()
    if len(df):
        df.to_csv(f"{RES}/permutation_importance.csv", index=False)
        for (c, m), sub in df.groupby(["cancer", "model"]):
            genes = ", ".join(sub.head(5).gene)
            print(f"  {c:5s} {m:9s} top-5: {genes}")
    return df


# --- 6j.2 robustness on every cohort ----------------------------------------
# Both reviews flagged that comparing only CoxRidge vs VAECox confounds
# "pretrained" with "deep nonlinear model with a bottleneck": a randomly
# initialised encoder of the same architecture would be expected to tolerate
# corrupted inputs better than a linear model regardless of pretraining. We
# therefore score every model that Phase 2 already trains, plus VAECox-Random
# (identical architecture, encoder never pretrained) so pretraining's own
# contribution can be read off directly against that ablation.
ROBUSTNESS_MODELS = ("CoxLasso", "CoxRidge", "Coxnnet", "CoxMLP", "VAECox", "VAECox-Random")


def robustness_all(cohorts=None, seeds=None, models=ROBUSTNESS_MODELS,
                   miss=(0.0, 0.1, 0.25, 0.5), sigmas=(0.0, 0.5, 1.0, 2.0)):
    """Corruption is applied at *inference* only, so each model is trained once
    per (cohort, seed) and then scored under every corruption level — the §6a
    version refit from scratch for every level, which was ~8x the compute for
    the same answer."""
    cohorts = cohorts or CFG["PAPER_10"]
    seeds = seeds if seeds is not None else CFG["SEEDS"][:5]
    rows = []
    for cohort in cohorts:
        acc = {}   # (experiment, level) -> {model: [cindex per seed]}
        for seed in seeds:
            set_seed(seed)
            tr, te = split_indices(cohort, seed)
            Xtr, Xte, ytr, yte, ctr, cte = split_arrays(cohort, tr, te)
            fitted = {m: fit_risk(m, Xtr, ytr, ctr, Xte)[0] for m in models}
            rng_m = np.random.default_rng(seed + 1000)
            rng_n = np.random.default_rng(seed + 2000)
            for frac in miss:
                Xc = Xte * (rng_m.random(Xte.shape) >= frac)
                for m, mod in fitted.items():
                    acc.setdefault(("missing", f"{int(frac*100)}%"), {}).setdefault(m, []).append(
                        cindex_safe(yte, _risk(mod, Xc.astype(np.float32)), cte))
            for sig in sigmas:
                Xc = (Xte + rng_n.normal(0, sig, Xte.shape)).astype(np.float32)
                for m, mod in fitted.items():
                    acc.setdefault(("noise", f"sigma={sig}"), {}).setdefault(m, []).append(
                        cindex_safe(yte, _risk(mod, Xc), cte))
        for (exp, lev), d in acc.items():
            row = dict(cancer=cohort, experiment=exp, level=lev)
            for m in models:
                row[m] = round(float(np.nanmean(d.get(m, [np.nan]))), 4)
            rows.append(row)
        print(f"  {cohort} done")
    df = pd.DataFrame(rows)
    df.to_csv(f"{RES}/robustness_by_cancer.csv", index=False)
    # Headline: relative degradation from the clean baseline, averaged over cohorts.
    for exp in ("missing", "noise"):
        sub = df[df.experiment == exp]
        if not len(sub): continue
        piv = sub.pivot_table(index="level", values=list(models), aggfunc="mean")
        clean = piv.iloc[0]
        rel = (piv - clean) / clean * 100
        print(f"\n  {exp}: % change in C-index vs clean (mean over cohorts)")
        print(rel.round(1).to_string())
    # Pretraining vs architecture: VAECox's advantage over its own untrained-
    # encoder twin, at the harshest corruption level of each experiment.
    if "VAECox" in df.columns and "VAECox-Random" in df.columns:
        for exp, lev in [("missing", f"{int(max(miss)*100)}%"), ("noise", f"sigma={max(sigmas)}")]:
            sub = df[(df.experiment == exp) & (df.level == lev)]
            if len(sub):
                delta = (sub["VAECox"] - sub["VAECox-Random"]).mean()
                print(f"  pretraining effect @ {exp}={lev}: "
                      f"VAECox - VAECox-Random = {delta:+.4f} C-index (mean over cohorts)")
    return df


# --- 6j.3 Kaplan-Meier for every cohort -------------------------------------
def kaplan_meier_all(cohorts=None, name="VAECox", seed=0):
    cohorts = cohorts or CFG["PAPER_10"]
    rows, panels = [], [c for c in cohorts if c in COHORT_DFS]
    ncol = min(5, max(1, len(panels)))
    nrow = int(np.ceil(len(panels) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 3.0 * nrow), squeeze=False)
    for ax, cohort in zip(axes.ravel(), panels):
        set_seed(seed)
        tr, te = split_indices(cohort, seed)
        Xtr, Xte, ytr, yte, ctr, cte = split_arrays(cohort, tr, te)
        _, risk = fit_risk(name, Xtr, ytr, ctr, Xte)
        hi = risk >= np.median(risk)
        ev = (cte == 0)
        kmf = KaplanMeierFitter()
        for grp, lab in [(hi, "High risk"), (~hi, "Low risk")]:
            if grp.sum() > 0:
                kmf.fit(yte[grp], ev[grp], label=lab); kmf.plot_survival_function(ax=ax, ci_show=False)
        p = float("nan")
        if hi.sum() > 0 and (~hi).sum() > 0:
            p = logrank_test(yte[hi], yte[~hi], ev[hi], ev[~hi]).p_value
        ax.set_title(f"{cohort}  p={p:.3g}", fontsize=9)
        ax.set_xlabel("Days"); ax.set_ylabel("S(t)"); ax.legend(fontsize=7)
        rows.append(dict(cancer=cohort, model=name, n_test=int(len(yte)),
                         n_events=int(ev.sum()), n_high=int(hi.sum()), n_low=int((~hi).sum()),
                         log_rank_p=round(p, 5) if not np.isnan(p) else None,
                         significant=bool(p < 0.05) if not np.isnan(p) else None))
    for ax in axes.ravel()[len(panels):]:
        ax.axis("off")
    fig.tight_layout(); fig.savefig(f'{CFG["OUT"]}/figures/km_all_cohorts.png', dpi=120); plt.close(fig)
    df = pd.DataFrame(rows)
    df.to_csv(f"{RES}/km_summary.csv", index=False)
    n_sig = int(df.significant.fillna(False).sum())
    print(f"  risk stratification significant (p<0.05) in {n_sig}/{len(df)} cohorts")
    return df


print("### 6j.1 permutation importance");       PERM_IMP = cached("permutation_importance", run_importance)
print("\n### 6j.2 robustness, every cohort");   ROB_ALL  = cached("robustness_by_cancer", robustness_all)
print("\n### 6j.3 Kaplan-Meier, every cohort"); KM_ALL   = cached("km_summary", kaplan_meier_all)

## 6k · Consolidated export

Every result above is dumped to `results/manuscript_numbers.json` — one file with
the dataset summary, training setup, the Phase 2 table and win counts, and all
extension results. Convenient for writing up without reopening eight CSVs.
Extension figures are written alongside it.

In [ ]:
# ---------------------------------------------------------------------------
# 6k · Export every number the manuscript needs, plus the extension figures.
# ---------------------------------------------------------------------------
def _safe(df, cols=None):
    if df is None or len(df) == 0:
        return []
    return df[cols].to_dict("records") if cols else df.to_dict("records")


NUMBERS = dict(
    data=dict(
        source="TCGA via GenoTEX / UCSC Xena (HiSeqV2_PANCAN)",
        n_cohorts_loaded=len(COHORT_DFS),
        cohorts=sorted(COHORT_DFS),
        evaluated=CFG["PAPER_10"],
        n_genes=NUM_FEATURES,
        per_cohort={c: dict(n=int(len(d)),
                            events=int((d.censored == 0).sum()),
                            censor_pct=round(100 * float((d.censored == 1).mean()), 1),
                            median_survival_days=float(np.median(d.survival)))
                    for c, d in COHORT_DFS.items()},
    ),
    setup=dict(device=str(DEVICE),
               gpu=torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU",
               vae_epochs=CFG["VAE_EPOCHS"], vae_hidden=CFG["HIDDEN"], vae_latent=CFG["LATENT"],
               vae_lr=CFG["VAE_LR"], vae_wd=CFG["VAE_WD"], vae_batch=CFG["VAE_BATCH"],
               surv_epochs=CFG["SURV_EPOCHS"], seeds=CFG["SEEDS"],
               neural_hp_grid=[list(h) for h in NEURAL_HP_GRID],
               linear_penalty_grid=list(PENALTY_GRID),
               pretrain_exclude_test=CFG["PRETRAIN_EXCLUDE_TEST"]),
    phase2=dict(table=_safe(PH2_LONG), wins=PH2_WINS,
                mean_cindex={m: (None if np.isnan(v) else round(float(v), 4))
                             for m, v in PH2_WIDE["Mean"].items()},
                best_mean_model=str(PH2_WIDE["Mean"].idxmax()),
                paper_wins="VAECox 7/10",
                paired_stats=_safe(PAIRED_STATS)),
    extensions=dict(
        robustness=_safe(ROB_ALL),
        subgroup=_safe(SUBGROUP),
        subgroup_gaps=_safe(GAPS),
        lightweight=_safe(LIGHT_CANCER),
        lightweight_disparity=_safe(LIGHT_GAP),
        feature_subset=_safe(FEATSUB),
        importance=_safe(PERM_IMP),
        kaplan_meier=_safe(KM_ALL),
    ),
)

with open(f"{RES}/manuscript_numbers.json", "w") as f:
    json.dump(NUMBERS, f, indent=2, default=str)
print(f"wrote {RES}/manuscript_numbers.json")


# ---- extension figures ------------------------------------------------------
FIG = f'{CFG["OUT"]}/figures'

# E1 · robustness curves, pooled over all cohorts
if len(ROB_ALL):
    _rob_model_cols = [c for c in ROBUSTNESS_MODELS if c in ROB_ALL.columns]
    pooled = ROB_ALL.groupby(["experiment", "level"], sort=False)[_rob_model_cols].mean()
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, exp in zip(axes, ["missing", "noise"]):
        sub = pooled.loc[exp] if exp in pooled.index.get_level_values(0) else None
        if sub is None: continue
        for m in sub.columns:
            ax.plot(range(len(sub)), sub[m], "-o",
                    label=m, lw=2.5 if m in ("VAECox", "VAECox-Random") else 1.2)
        ax.set_xticks(range(len(sub))); ax.set_xticklabels(sub.index, rotation=20)
        ax.set_ylabel("C-index"); ax.legend(fontsize=7)
        ax.set_title(f"Robustness — {exp} (mean over {ROB_ALL.cancer.nunique()} cohorts)")
    fig.tight_layout(); fig.savefig(f"{FIG}/ext_robustness_all.png", dpi=120); plt.close(fig)

# E2 · lightweight trade-off: params vs C-index, sized by train time
if len(LIGHT_CANCER):
    agg = LIGHT_CANCER.groupby("config").agg(
        n_params=("n_params", "first"), train_sec=("train_sec", "first"),
        mean_cindex=("cindex", "mean")).reset_index()
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    ax.scatter(agg.n_params / 1e6, agg.mean_cindex,
               s=30 + 4 * agg.train_sec, alpha=.7, color="#4C78A8")
    for _, r in agg.iterrows():
        ax.annotate(r.config, (r.n_params / 1e6, r.mean_cindex),
                    textcoords="offset points", xytext=(6, 4), fontsize=8)
    ax.set_xscale("log"); ax.set_xlabel("VAE parameters (millions, log scale)")
    ax.set_ylabel("Mean C-index"); ax.set_title("Lightweight VAEs — size vs accuracy (bubble = train seconds)")
    fig.tight_layout(); fig.savefig(f"{FIG}/ext_lightweight.png", dpi=120); plt.close(fig)

# E3 · subgroup gaps
if len(GAPS):
    top = GAPS.sort_values("gap", ascending=False).head(15)[::-1]
    fig, ax = plt.subplots(figsize=(7.5, 5))
    ax.barh([f"{r.cancer} · {r.variable} ({r.model})" for _, r in top.iterrows()],
            top.gap, color="#E45756")
    ax.set_xlabel("C-index gap (best group − worst group)")
    ax.set_title("Largest within-cohort subgroup disparities")
    fig.tight_layout(); fig.savefig(f"{FIG}/ext_subgroup_gaps.png", dpi=120); plt.close(fig)

# E4 · feature-subset accessibility
if len(FEATSUB):
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    for name, sub in FEATSUB.groupby("model"):
        sub = sub.sort_values("k_genes")
        ax.plot(sub.k_genes, sub.mean_cindex, "-o", label=name)
    ax.set_xscale("log"); ax.set_xlabel("Number of genes retained (log scale)")
    ax.set_ylabel("Mean C-index"); ax.legend()
    ax.set_title("Low-resource accessibility — C-index vs feature budget")
    fig.tight_layout(); fig.savefig(f"{FIG}/ext_feature_subset.png", dpi=120); plt.close(fig)

# E5 · lightweight disparity: does shrinking hurt small cohorts more?
if len(LIGHT_GAP):
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    for cfg_name, sub in LIGHT_GAP.groupby("config"):
        ax.scatter(sub.n_events, sub.delta_vs_full, label=cfg_name, alpha=.8)
    ax.axhline(0, color="grey", lw=1, ls="--")
    ax.set_xlabel("Uncensored events in cohort")
    ax.set_ylabel("Δ C-index vs full-size VAE")
    ax.legend(); ax.set_title("Do lightweight models hurt small cohorts more?")
    fig.tight_layout(); fig.savefig(f"{FIG}/ext_lightweight_disparity.png", dpi=120); plt.close(fig)

print("\nExtension figures written to", FIG)
for fn in sorted(os.listdir(FIG)):
    print("  ", fn)

## 7 · Figures & reproducibility card

In [ ]:
MODEL_ORDER = [m for m in ["CoxLasso", "CoxRidge", "Coxnnet", "CoxMLP", "VAECox"]
               if m in PH2_WIDE.index]
CANCERS = [c for c in PH2_WIDE.columns if c != "Mean"]

_paired_line = "not computed"
if len(PAIRED_STATS):
    _sig = PAIRED_STATS[PAIRED_STATS.wilcoxon_p < 0.05]
    _paired_line = (f"{len(_sig)}/{len(PAIRED_STATS)} (cancer, baseline) pairs "
                    f"significant at p<0.05 (Wilcoxon signed-rank on per-seed "
                    f"VAECox-minus-baseline differences); see paired_stats.csv")
PIV_M = PH2_LONG.pivot(index="model", columns="cancer", values="mean_cindex").reindex(MODEL_ORDER)[CANCERS]
PIV_S = PH2_LONG.pivot(index="model", columns="cancer", values="std_cindex").reindex(MODEL_ORDER)[CANCERS]

# F1 · per-cancer C-index WITH seed variability — the headline result.
# The mean-only bar chart further down hides that model gaps are often smaller
# than the seed-to-seed spread, which is exactly what a reader needs to judge it.
x, w = np.arange(len(CANCERS)), 0.8 / max(1, len(MODEL_ORDER))
fig, ax = plt.subplots(figsize=(1.15 * len(CANCERS) + 3, 4.5))
for i, m in enumerate(MODEL_ORDER):
    ax.bar(x + i * w - 0.4 + w / 2, PIV_M.loc[m], w, yerr=PIV_S.loc[m], capsize=2,
           label=m, color="#E45756" if m == "VAECox" else None)
ax.axhline(0.5, color="grey", ls="--", lw=1)
ax.annotate("random (0.5)", (len(CANCERS) - 0.5, 0.5), fontsize=7, color="grey",
            va="bottom", ha="right")
ax.set_xticks(x); ax.set_xticklabels(CANCERS, rotation=45, ha="right")
ax.set_ylabel("C-index (mean ± sd over seeds)")
ax.set_title("Phase 2 — C-index per cancer type")
ax.legend(ncol=len(MODEL_ORDER), fontsize=8, loc="upper left")
fig.tight_layout(); fig.savefig(f'{CFG["OUT"]}/figures/ph2_cindex_per_cancer.png', dpi=120)
plt.close(fig)

# F2 · the same numbers as a heatmap — easier to scan for who wins where
fig, ax = plt.subplots(figsize=(1.0 * len(CANCERS) + 3, 0.6 * len(MODEL_ORDER) + 2))
vals = PIV_M.values.astype(float)
im = ax.imshow(vals, cmap="RdYlBu_r", aspect="auto",
               vmin=min(0.40, np.nanmin(vals)), vmax=max(0.70, np.nanmax(vals)))
ax.set_xticks(range(len(CANCERS))); ax.set_xticklabels(CANCERS, rotation=45, ha="right")
ax.set_yticks(range(len(MODEL_ORDER))); ax.set_yticklabels(MODEL_ORDER)
for i in range(vals.shape[0]):
    for j in range(vals.shape[1]):
        if not np.isnan(vals[i, j]):
            ax.text(j, i, f"{vals[i, j]:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax, label="C-index", shrink=.8)
ax.set_title("Phase 2 — C-index heatmap")
fig.tight_layout(); fig.savefig(f'{CFG["OUT"]}/figures/ph2_cindex_heatmap.png', dpi=120)
plt.close(fig)

# F3 · win counts against the paper's 7/10 claim
fig, ax = plt.subplots(figsize=(6, 3.5))
wins = [PH2_WINS.get(m, 0) for m in MODEL_ORDER]
ax.bar(MODEL_ORDER, wins,
       color=["#E45756" if m == "VAECox" else "#B0B8C4" for m in MODEL_ORDER])
ax.axhline(7, color="#4C78A8", ls="--", lw=1.5)
ax.annotate("paper: VAECox 7/10", (len(MODEL_ORDER) - 0.4, 7), fontsize=8,
            color="#4C78A8", va="bottom", ha="right")
ax.set_ylabel(f"Cancers won (of {len(CANCERS)})")
ax.set_ylim(0, max(8, max(wins) + 1))
ax.set_title("Phase 2 — per-cancer wins")
fig.tight_layout(); fig.savefig(f'{CFG["OUT"]}/figures/ph2_wins.png', dpi=120); plt.close(fig)

# Bar chart of mean C-index per model
fig, ax = plt.subplots(figsize=(7, 4))
PH2_WIDE["Mean"].sort_values().plot.barh(ax=ax, color="#4C78A8")
ax.set_xlabel("Mean C-index"); ax.set_title("VAECox reproduction — mean C-index across 10 cancers")
fig.tight_layout(); fig.savefig(f'{CFG["OUT"]}/figures/mean_cindex.png', dpi=120); plt.close(fig)

# Robustness figure — prefers the all-cohort table; §6a's single cohort is a
# fallback for when SKIP_SUPERSEDED is off and you want the STAD-only view.
_rob_src = ROB_ALL if len(ROB_ALL) else ROB
fig, ax = plt.subplots(figsize=(7, 4))
miss = (_rob_src[_rob_src.experiment == "missing"]
        .groupby("level", sort=False)[["CoxRidge", "VAECox"]].mean().reset_index()
        if len(_rob_src) else pd.DataFrame(columns=["level", "CoxRidge", "VAECox"]))
ax.plot(range(len(miss)), miss.CoxRidge, "-o", label="CoxRidge")
ax.plot(range(len(miss)), miss.VAECox, "-o", label="VAECox")
ax.set_xticks(range(len(miss))); ax.set_xticklabels(miss.level)
ax.set_xlabel("Missing features"); ax.set_ylabel("C-index"); ax.legend()
ax.set_title("Robustness to missing features")
fig.tight_layout(); fig.savefig(f'{CFG["OUT"]}/figures/robustness.png', dpi=120); plt.close(fig)


# ---- extension summary lines, computed rather than typed --------------------
def _fmt_wins():
    return ", ".join(f"{m} {w}" for m, w in sorted(PH2_WINS.items(), key=lambda x: -x[1]))


_gap_line = "not computed"
if len(GAPS):
    worst = GAPS.iloc[0]
    _gap_line = (f"largest gap {worst.gap:.3f} C-index ({worst.cancer}/{worst.variable}, "
                 f"{worst.model}: {worst.best_group} vs {worst.worst_group}); "
                 f"mean gap across strata {GAPS.gap.mean():.3f}")

_light_line = "not computed"
if len(LIGHT_GAP):
    med = LIGHT_GAP.n_events.median()
    sm = LIGHT_GAP[LIGHT_GAP.n_events <= med].delta_vs_full.mean()
    lg = LIGHT_GAP[LIGHT_GAP.n_events > med].delta_vs_full.mean()
    _light_line = (f"shrinking the VAE changes C-index by {sm:+.3f} on low-event cohorts "
                   f"vs {lg:+.3f} on high-event cohorts")

_rob_line = "not computed"
if len(ROB_ALL):
    parts = []
    for exp, lev in [("missing", "50%"), ("noise", "sigma=2.0")]:
        s = ROB_ALL[(ROB_ALL.experiment == exp) & (ROB_ALL.level == lev)]
        c0 = ROB_ALL[ROB_ALL.experiment == exp].groupby("level").mean(numeric_only=True).iloc[0]
        if len(s):
            parts.append(f"{exp}={lev}: CoxRidge {s.CoxRidge.mean():.3f} (from {c0.CoxRidge:.3f}), "
                         f"VAECox {s.VAECox.mean():.3f} (from {c0.VAECox:.3f})")
    _rob_line = "; ".join(parts) if parts else "not computed"

_km_line = ("not computed" if not len(KM_ALL) else
            f"{int(KM_ALL.significant.fillna(False).sum())}/{len(KM_ALL)} cohorts show "
            f"significant high/low risk separation (log-rank p<0.05)")

_cpu_line = "not computed"
if len(FEATSUB):
    cpu_rows = FEATSUB[FEATSUB.device == "cpu"]
    if len(cpu_rows):
        r = cpu_rows.iloc[0]
        _cpu_line = (f"k={int(r.k_genes)} genes, {r.vae_params/1e6:.2f}M-param VAE trains on CPU in "
                     f"{r.vae_pretrain_sec}s; C-index {r.mean_cindex}")

card = f"""
================================================================================
REPRODUCIBILITY CARD — VAECox (Bioinformatics 2020, Suppl. 1)
================================================================================
Paper : Kim, Kim, Choe, Lee, Kang. "Improved survival analysis by learning
        shared genomic information from pan-cancer data."
        Bioinformatics 36(Suppl_1):i389-i398. DOI:10.1093/bioinformatics/btaa462
Claim : VAECox outperforms CoxLasso/CoxRidge/Coxnnet on 7/10 TCGA cancers (C-index).

DATA SOURCE     : REAL TCGA via GenoTEX (UCSC Xena TCGA Hub, HiSeqV2_PANCAN)
Cohorts         : {sorted(COHORT_DFS)}
Genes (VAE dim) : {NUM_FEATURES}
Device          : {DEVICE} ({torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else 'CPU'})
Seeds           : {CFG['SEEDS']}

VAE             : {NUM_FEATURES}->{CFG['HIDDEN']}->{CFG['LATENT']} (mu,sigma), Tanh, Adam
                  lr={CFG['VAE_LR']} wd={CFG['VAE_WD']} epochs={CFG['VAE_EPOCHS']} batch={CFG['VAE_BATCH']}
VAECox          : pretrained encoder FINE-TUNED + Coxnnet(128)  [paper's method]
HP search       : neural {'grid ' + str(NEURAL_HP_GRID) if CFG['HP_SEARCH'] else 'none'} per cancer;
                  linear (CoxLasso/CoxRidge) penalty path {str(PENALTY_GRID) if CFG['HP_SEARCH'] else 'none'}
VAE pretraining : eval-cohort test patients {'EXCLUDED (union over all seeds)' if CFG['PRETRAIN_EXCLUDE_TEST'] else 'NOT excluded (legacy/leaky)'}
Surv epochs     : {CFG['SURV_EPOCHS']}

--------------------------------------------------------------------------------
PHASE 2 — REPRODUCTION
--------------------------------------------------------------------------------
Wins (this run) : {_fmt_wins()}
Paper wins      : VAECox 7/10
Mean C-index    : {', '.join(f'{m} {v:.3f}' for m, v in PH2_WIDE['Mean'].items())}
Best mean       : {PH2_WIDE['Mean'].idxmax()}
Paired stats    : {_paired_line}

--------------------------------------------------------------------------------
PHASE 3 — EXTENSIONS
--------------------------------------------------------------------------------
Subgroup fairness   : {_gap_line}
Lightweight equity  : {_light_line}
Robustness          : {_rob_line}
Risk stratification : {_km_line}
CPU accessibility   : {_cpu_line}
Interpretability    : permutation importance over {PERM_IMP.cancer.nunique() if len(PERM_IMP) else 0} cohorts
                      x {PERM_IMP.model.nunique() if len(PERM_IMP) else 0} models

--------------------------------------------------------------------------------
DEVIATIONS
--------------------------------------------------------------------------------
  - HP search reduced vs paper's 18-combo 5-fold CV (time). Documented.
  - Extensions use fixed HP ({DEFAULT_HP}) rather than the per-cancer search,
    so extension C-indices are not directly comparable to the Phase 2 table.
  - VAE minibatched on GPU (paper full-batch); numerically equivalent.
  - Lightweight sweep pretrains every config for {SWEEP_VAE_EPOCHS} epochs (not
    {CFG['VAE_EPOCHS']}) so configs are compared at an equal compute budget.
  - Robustness (6j.2) uses {len(CFG['SEEDS'][:5])} seeds; subgroup fairness (6h) uses all {len(CFG['SEEDS'])}.
  - Subgroup gaps exclude TCGA free-text histology placeholders ("Other, specify"
    / "Mixed Histology (please specify)") — not clinical categories.
  - GenoTEX HiSeqV2_PANCAN gene set ({NUM_FEATURES}) may differ slightly from
    paper's 20,502 (pan-cancer-normalised vs per-cohort). Documented.
  - Subgroup strata below {MIN_GROUP_N} patients or {MIN_GROUP_EV} events are dropped as
    unestimable rather than reported.
================================================================================
"""
with open(f'{CFG["OUT"]}/results/reproducibility_card.txt', "w") as f:
    f.write(card)
print(card)


## 8 · What you end up with

Phases 2 and 3 are complete once this notebook finishes. §9 bundles the outputs
into a single zip:

* `results/cindex_long.csv`, `cindex_comparison.csv` — the C-index tables and win counts
* `results/cindex_by_seed.csv`, `paired_stats.csv` — per-seed values and the paired
  VAECox-vs-baseline comparison (win count, Wilcoxon signed-rank, bootstrap CI)
* `results/*.csv` — every extension result (robustness, subgroup fairness,
  lightweight sweep, feature budget, permutation importance, Kaplan–Meier)
* `results/manuscript_numbers.json` — all of the above consolidated into one file
* `results/reproducibility_card.txt` — settings, seeds, deviations
* `figures/*.png` — 14 figures

## 9 · Package everything for download

`/kaggle/working` also holds the ~680 MB VAE checkpoint, so "Download all" would
hand you a huge archive that is mostly a file you do not need (it is gitignored
anyway, and it stays in the Kaggle output so a re-run can resume from it).

This cell writes **one small zip** with only what the repo and the manuscript
need — CSVs, figures, the card, and `manuscript_numbers.json` — and checks that
every expected output actually exists, so a section that silently failed cannot
slip past unnoticed.

In [ ]:
import zipfile

BUNDLE = f'{CFG["OUT"]}/vaecox_results.zip'
KEEP_EXT = {".csv", ".png", ".txt", ".json"}

# Everything the manuscript build and the repo expect. Missing entries are
# reported rather than silently tolerated.
EXPECTED = [
    "results/cindex_long.csv", "results/cindex_comparison.csv",
    "results/manuscript_numbers.json", "results/reproducibility_card.txt",
    "results/robustness_by_cancer.csv",
    "results/fairness.csv", "results/cohort_fairness.csv",
    "results/subgroup_cindex.csv", "results/subgroup_gaps.csv",
    "results/lightweight_by_cancer.csv",
    "results/lightweight_disparity.csv", "results/feature_subset.csv",
    "results/subgroup_excluded_strata.csv",
    "results/permutation_importance.csv", "results/km_summary.csv",
    "figures/ph2_cindex_per_cancer.png", "figures/ph2_cindex_heatmap.png",
    "figures/ph2_wins.png", "figures/mean_cindex.png", "figures/robustness.png",
    "figures/km_all_cohorts.png", "figures/ext_robustness_all.png",
    "figures/ext_lightweight.png", "figures/ext_subgroup_gaps.png",
    "figures/ext_feature_subset.png", "figures/ext_lightweight_disparity.png",
]
if not CFG["SKIP_SUPERSEDED"]:
    # Only §6a/§6c write these; with SKIP_SUPERSEDED on they are absent by design
    # and must not be reported as failures.
    EXPECTED += ["results/robustness.csv", "results/lightweight.csv"]

included, excluded = [], []
if os.path.exists(BUNDLE):
    os.remove(BUNDLE)
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(CFG["OUT"]):
        for fn in sorted(files):
            full = os.path.join(root, fn)
            rel = os.path.relpath(full, CFG["OUT"])
            if os.path.abspath(full) == os.path.abspath(BUNDLE):
                continue
            if os.path.splitext(fn)[1].lower() in KEEP_EXT:
                z.write(full, rel)
                included.append((rel, os.path.getsize(full)))
            else:
                excluded.append((rel, os.path.getsize(full)))

print(f"BUNDLE: {BUNDLE}  ({os.path.getsize(BUNDLE)/1e6:.1f} MB, {len(included)} files)\n")
for rel, sz in sorted(included):
    print(f"  {sz/1024:8.1f} KB  {rel}")

if excluded:
    print("\nExcluded (not needed downstream):")
    for rel, sz in sorted(excluded):
        print(f"  {sz/1e6:8.1f} MB  {rel}")

have = {rel for rel, _ in included}
missing = [e for e in EXPECTED if e not in have]
print()
if missing:
    print(f"!! {len(missing)} EXPECTED OUTPUT(S) MISSING — a section did not run:")
    for e in missing:
        print("   ", e)
else:
    print("All expected outputs present.")

print(f"""
NEXT: Kaggle right panel -> Output -> download {os.path.basename(BUNDLE)}, then:

    unzip {os.path.basename(BUNDLE)} -d out/
""")
